# 01: Your First API Call

You already use Claude in two ways: **Claude chat** (the web/app interface) and **Claude Code** (the CLI agent in your terminal). This notebook introduces a third way — the **Claude API** — which is the key to building custom research tools that work exactly the way you need them to.

By the end of this notebook, you'll be able to send messages to Claude programmatically from Python and understand exactly what comes back.

> **Navigation:** [← Previous: Probability and AI Math](../07-math-you-need/03-probability-and-ai-math.ipynb) | [Module Index](../README.md) | [Next: Structured Outputs →](../08-claude-api/02-structured-outputs.ipynb)

---

## API vs Chat vs Claude Code — when to use which

| Tool | Best for | Example |
|------|----------|--------|
| **Claude chat** | One-off questions, brainstorming, exploring ideas | "What's the current understanding of NaV1.7's role in inflammatory pain?" |
| **Claude Code** | Working with files, writing code, editing projects | "Add error bars to this calcium imaging plot" |
| **Claude API** | Automation, batch processing, building custom tools | Process 200 paper abstracts and extract targets + mechanisms |

The API is what you reach for when you need Claude to do the **same kind of task many times**, or when you want to **embed Claude inside a larger workflow**. Think of it as the difference between pipetting by hand (chat) and setting up a plate reader protocol (API).

---

## Setting up your API key

The API authenticates with an **API key** — a secret string that identifies your account. You get one from [console.anthropic.com](https://console.anthropic.com/).

The safe way to provide it is via an **environment variable** called `ANTHROPIC_API_KEY`. You should have this set already (Claude Code uses the same key). Let's verify:

In [ ]:
import os

# Check that the API key is available
api_key = os.environ.get("ANTHROPIC_API_KEY")

if api_key:
    # Only show first 8 and last 4 characters — never print the full key
    print(f"API key found: {api_key[:8]}...{api_key[-4:]}")
else:
    print("WARNING: ANTHROPIC_API_KEY not set!")
    print("Set it in your terminal before launching Jupyter:")
    print('  export ANTHROPIC_API_KEY="sk-ant-..."')

> **Security rule:** Never paste your API key directly into a notebook or Python file. Always use environment variables. If you accidentally commit a key to git, rotate it immediately at console.anthropic.com.

---

## The Anthropic Python SDK

The `anthropic` package is already installed in our environment. Let's import it and create a **client** — the object we'll use for all API calls.

In [ ]:
from anthropic import Anthropic

# Create the client — it automatically reads ANTHROPIC_API_KEY from the environment
client = Anthropic()

print("Client ready.")

That's all the setup. Two lines: import and create. Now let's make our first call.

---

## Anatomy of an API call

Every API call has the same basic structure:

```python
message = client.messages.create(
    model="claude-sonnet-4-6",         # which Claude model to use
    max_tokens=1024,                    # maximum length of the response
    system="...",                       # optional: system prompt (sets behavior)
    messages=[                           # the conversation
        {"role": "user", "content": "your question here"}
    ]
)
```

Let's break down each parameter:

| Parameter | What it does |
|-----------|-------------|
| `model` | Which Claude model to use. We'll use `claude-sonnet-4-6` — fast and cost-effective for most tasks. |
| `max_tokens` | Cap on response length. 1024 tokens is ~750 words — plenty for most answers. |
| `system` | Sets Claude's role/behavior for the whole conversation. Optional but very useful. |
| `messages` | A list of messages. Each has a `role` ("user" or "assistant") and `content`. |

### API Call Lifecycle

```mermaid
sequenceDiagram
    participant You as Your Python Code
    participant SDK as Anthropic SDK
    participant HTTP as HTTPS Request
    participant API as Anthropic Servers
    
    You->>SDK: client.messages.create(model, messages, ...)
    SDK->>HTTP: POST /v1/messages<br/>+ API key header<br/>+ JSON body
    HTTP->>API: Encrypted request<br/>over the internet
    
    Note over API: Claude processes your prompt<br/>generates response tokens
    
    API->>HTTP: JSON response<br/>(content, usage, stop_reason)
    HTTP->>SDK: Parse HTTP response
    SDK->>You: Message object<br/>.content[0].text = "..."<br/>.usage.input_tokens = N<br/>.usage.output_tokens = M
```

Each API call follows this lifecycle. Your code never communicates with Claude directly -- the SDK handles authentication, serialization, error handling, and retries. The response comes back as a structured `Message` object that you can inspect for both the text content and metadata like token usage.

---

## Your first call: ask Claude about a pain biology concept

Let's ask Claude to explain a concept relevant to your work.

In [ ]:
message = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": "Explain in 3 sentences how NaV1.7 gain-of-function mutations cause pain disorders."
        }
    ]
)

# Extract the text from the response
print(message.content[0].text)

That's it. You just called an LLM from Python. Let's understand what came back.

---

## Understanding the response object

The `message` variable isn't just text — it's a structured object with useful metadata.

In [ ]:
# The full response object
print("Type:", type(message))
print("Model used:", message.model)
print("Stop reason:", message.stop_reason)
print()

# Token usage — this is what determines cost
print("Input tokens:", message.usage.input_tokens)
print("Output tokens:", message.usage.output_tokens)
print()

# The actual content
print("Number of content blocks:", len(message.content))
print("Content type:", message.content[0].type)
print("Text:", message.content[0].text[:100], "...")

Key fields:

- **`message.content[0].text`** — the actual response text. This is what you'll use 99% of the time.
- **`message.usage`** — token counts, which determine cost.
- **`message.stop_reason`** — why Claude stopped. `"end_turn"` means it finished naturally. `"max_tokens"` means it was cut off (you should increase `max_tokens`).
- **`message.model`** — confirms which model was used.

---

## Adding a system prompt

The **system prompt** tells Claude how to behave. It's like briefing a research assistant before they start. This is where you set the expertise level, output format, and constraints.

In [ ]:
message = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    system="You are an expert pain neurobiologist. Give precise, technical answers "
           "suitable for a researcher who designs de novo protein binders targeting "
           "ion channels. Keep answers concise — no more than 4 sentences.",
    messages=[
        {
            "role": "user",
            "content": "What makes the NaV1.8 channel a promising target for pain therapeutics "
                       "compared to NaV1.7?"
        }
    ]
)

print(message.content[0].text)

Notice how the system prompt shapes the response — you get a technical, focused answer instead of a broad textbook explanation. The system prompt is your most powerful lever for controlling Claude's output.

---

## A reusable helper function

You'll be making lots of API calls. Let's wrap the pattern in a function so we don't repeat ourselves.

In [ ]:
def ask_claude(prompt, system=None, model="claude-sonnet-4-6", max_tokens=1024):
    """Send a prompt to Claude and return the response text."""
    kwargs = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": prompt}]
    }
    if system:
        kwargs["system"] = system
    
    message = client.messages.create(**kwargs)
    return message.content[0].text


# Test it
answer = ask_claude(
    "What is TRPV1 and why is it important in nociception?",
    system="Answer in exactly 2 sentences."
)
print(answer)

Now you can call `ask_claude("your question")` anywhere in this notebook (or copy the function into other notebooks).

> **Think Python connection:** This is the same function pattern from Module 01 — `def`, parameters with defaults, a docstring, and `return`. The only new thing is that the function body calls an external API instead of doing math.

---

## Cost awareness

API calls cost money. Not much per call, but it adds up if you're processing thousands of items. Here's what you need to know:

### Models and pricing (approximate, as of early 2026)

| Model | Input (per 1M tokens) | Output (per 1M tokens) | When to use |
|-------|----------------------|------------------------|-------------|
| `claude-sonnet-4-6` | ~$3 | ~$15 | **Default choice.** Great for most tasks. |
| `claude-opus-4-6` | ~$15 | ~$75 | Complex reasoning, nuanced analysis. |
| `claude-haiku-3-5` | ~$0.80 | ~$4 | High-volume, simpler tasks (classification, extraction). |

### Token intuition

- 1 token is roughly 4 characters or 0.75 words
- A typical abstract: ~200-300 tokens
- A page of text: ~400-500 tokens
- Your question + Claude's answer in the example above: check the usage below

In [ ]:
# Let's check the cost of our earlier call
# We'll make a fresh call and inspect usage

message = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=512,
    messages=[{"role": "user", "content": "What is the role of SCN9A in inherited pain disorders? Answer in 3 sentences."}]
)

inp = message.usage.input_tokens
out = message.usage.output_tokens

# Approximate cost for claude-sonnet-4-6
cost_input = inp * 3.0 / 1_000_000
cost_output = out * 15.0 / 1_000_000
total = cost_input + cost_output

print(f"Input tokens:  {inp}")
print(f"Output tokens: {out}")
print(f"Estimated cost: ${total:.5f}")
print(f"\nAt this rate, processing 1000 similar questions would cost ~${total * 1000:.2f}")

print(f"\n{message.content[0].text}")

**Rule of thumb:** For the batch-processing tasks you'll build in this module, a run of a few hundred items will typically cost well under $1. Just stay aware of it — especially if you're using Opus or sending large inputs.

---

## Choosing max_tokens wisely

Setting `max_tokens` too low cuts off responses. Setting it too high doesn't cost extra (you only pay for tokens actually generated), but it's good practice to set a reasonable cap. Some guidelines:

| Task | Suggested max_tokens |
|------|---------------------|
| Short answer (1-3 sentences) | 256 |
| Paragraph explanation | 512-1024 |
| Detailed analysis | 2048-4096 |
| Long-form generation | 4096-8192 |

In [ ]:
# What happens when max_tokens is too low?
message = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=20,  # way too low for a real answer
    messages=[{"role": "user", "content": "Describe the mechanism of TRPV1 activation."}]
)

print("Stop reason:", message.stop_reason)  # will be "max_tokens" instead of "end_turn"
print("Response:", message.content[0].text)

When `stop_reason` is `"max_tokens"`, Claude was cut off mid-answer. Always check this if your responses look incomplete.

---

## Exercise: Gene summary function

Write a function called `gene_pain_summary` that:

1. Takes a **gene name** (e.g., `"SCN9A"`, `"TRPV1"`, `"CALCA"`)
2. Sends it to Claude with a system prompt that asks for a structured summary of the gene's role in pain biology
3. Returns the text response

Then call it for at least 3 different pain-related genes and print the results.

Hint: use the `ask_claude` helper function we built above, or write the `client.messages.create` call directly — either approach works.

In [ ]:
# Your code here

def gene_pain_summary(gene_name):
    """Get a summary of a gene's role in pain biology from Claude."""
    system = (
        "You are an expert pain neurobiologist. When given a gene name, provide a brief "
        "summary covering: (1) what protein it encodes, (2) where it's expressed in the "
        "pain pathway, (3) its role in nociception, and (4) any known link to human pain "
        "disorders. Keep it to 4-5 sentences."
    )
    return ask_claude(
        f"Summarize the role of {gene_name} in pain biology.",
        system=system
    )


# Test with pain-relevant genes
genes = ["SCN9A", "SCN10A", "TRPV1"]

for gene in genes:
    print(f"=== {gene} ===")
    print(gene_pain_summary(gene))
    print()

---

## Exercise: Experiment with parameters

Try modifying the API call to see how different parameters change the output:

1. Change the system prompt to request a different format (e.g., bullet points instead of sentences)
2. Try different `max_tokens` values and observe how `stop_reason` changes
3. Compare the same question with and without a system prompt

In [ ]:
# Your experiments here

# Example: bullet-point format
answer = ask_claude(
    "What are the main classes of nociceptors in DRG?",
    system="Answer using bullet points. Be concise — one line per bullet."
)
print(answer)

---

## What you just learned

- **API vs chat vs Claude Code** — the API is for automation and custom tools
- **Authentication** — `ANTHROPIC_API_KEY` environment variable, never hardcoded
- **The SDK** — `from anthropic import Anthropic`, then `client = Anthropic()`
- **Making a call** — `client.messages.create(model, max_tokens, messages)`
- **System prompts** — control Claude's behavior and output format
- **The response object** — `message.content[0].text` for the answer, `message.usage` for token counts
- **Cost awareness** — tokens drive cost; Sonnet is the cost-effective default

Next up: getting **structured data** (JSON) from Claude instead of free text — which is what makes the API truly powerful for research pipelines.

> **Navigation:** [← Previous: Probability and AI Math](../07-math-you-need/03-probability-and-ai-math.ipynb) | [Module Index](../README.md) | [Next: Structured Outputs →](../08-claude-api/02-structured-outputs.ipynb)

---

## Edit Log

- 2026-03-25: Created notebook with initial content
- 2026-03-25: Added cross-module navigation links